In [ ]:

!pip -q install kagglehub

import kagglehub

path = kagglehub.dataset_download(
    "rmisra/imdb-spoiler-dataset"
)

print("Dataset downloaded to:", path)


100%|██████████| 331M/331M [00:02<00:00, 154MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/rmisra/imdb-spoiler-dataset/versions/1


In [ ]:

import os

print("Dataset folder:", path)
print("Files:", os.listdir(path))


Dataset folder: /root/.cache/kagglehub/datasets/rmisra/imdb-spoiler-dataset/versions/1
Files: ['IMDB_reviews.json', 'IMDB_movie_details.json']


In [ ]:

import pandas as pd
import os

file_path = os.path.join(path, "IMDB_reviews.json")

reviews = pd.read_json(
    file_path,
    lines=True,
    nrows=1000
)

reviews.head()


,review_date,movie_id,user_id,is_spoiler,review_text,rating,review_summary
0,10 February 2006,tt0111161,ur1898687,True,"In its Oscar year, Shawshank Redemption (writt...",10,A classic piece of unforgettable film-making.
1,6 September 2000,tt0111161,ur0842118,True,The Shawshank Redemption is without a doubt on...,10,Simply amazing. The best film of the 90's.
2,3 August 2001,tt0111161,ur1285640,True,I believe that this film is the best story eve...,8,The best story ever told on film
3,1 September 2002,tt0111161,ur1003471,True,"**Yes, there are SPOILERS here**This film has ...",10,Busy dying or busy living?
4,20 May 2004,tt0111161,ur0226855,True,At the heart of this extraordinary movie is a ...,8,"Great story, wondrously told and acted"


In [ ]:

print(reviews.shape)

print(reviews['is_spoiler'].value_counts())


(1000, 7)
is_spoiler
True     956
False     44
Name: count, dtype: int64


In [ ]:

import pandas as pd
import os

file_path = os.path.join(path, "IMDB_reviews.json")

spoilers = []
non_spoilers = []

# Read the dataset in smaller chunks
for chunk in pd.read_json(
    file_path,
    lines=True,
    chunksize=10000
):
    chunk = chunk[['review_text', 'is_spoiler']].dropna()

    spoilers.append(
        chunk[chunk['is_spoiler'] == True].head(1000)
    )

    non_spoilers.append(
        chunk[chunk['is_spoiler'] == False].head(1000)
    )

# Combine the reviews
spoilers = pd.concat(spoilers).head(1000)
non_spoilers = pd.concat(non_spoilers).head(1000)

reviews = pd.concat([spoilers, non_spoilers])

# Mix the rows randomly
reviews = reviews.sample(frac=1, random_state=42)

print(reviews.shape)
print(reviews['is_spoiler'].value_counts())


(2000, 2)
is_spoiler
False    1000
True     1000
Name: count, dtype: int64


In [ ]:

from sklearn.model_selection import train_test_split

# X = Movie review text (input)
X = reviews['review_text']

# y = Spoiler or not (correct answer)
y = reviews['is_spoiler']

# Split into 75% training and 25% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))


Training reviews: 1500
Testing reviews: 500


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# Create our AI model
model = make_pipeline(
    TfidfVectorizer(max_features=5000),
    LogisticRegression(max_iter=1000)
)

# Train the AI using 1500 reviews
model.fit(X_train, y_train)

print("AI model trained successfully!")


AI model trained successfully!


In [ ]:

from sklearn.metrics import accuracy_score, classification_report

# Predict labels for the 500 test reviews
predictions = model.predict(X_test)

# Compare predictions with correct answers
accuracy = accuracy_score(y_test, predictions)

print("Model accuracy:", round(accuracy * 100, 2), "%")
print()
print(classification_report(y_test, predictions))


Model accuracy: 70.8 %

              precision    recall  f1-score   support

       False       0.69      0.76      0.72       250
        True       0.73      0.66      0.69       250

    accuracy                           0.71       500
   macro avg       0.71      0.71      0.71       500
weighted avg       0.71      0.71      0.71       500



In [ ]:

# Ask the user to enter a movie review
review = input("Enter a movie review: ")

# Ask our trained AI to predict
prediction = model.predict([review])

# Display the result
if prediction[0] == True:
    print("SPOILER ALERT! This review may contain spoilers.")
else:
    print("SAFE! No spoiler detected.")


Enter a movie review: the father of the main character dies 
SAFE! No spoiler detected.


In [ ]:

# Check predictions for our test reviews
from sklearn.metrics import confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions, labels=[False, True]))

print("\nPrediction counts:")
print(pd.Series(predictions).value_counts())


Confusion Matrix:
[[189  61]
 [ 85 165]]

Prediction counts:
False    274
True     226
Name: count, dtype: int64


In [ ]:

examples = [
    "The acting was amazing and the soundtrack was beautiful.",
    "At the end of the movie, the detective discovers that his best friend was the killer."
]

probabilities = model.predict_proba(examples)

for review, probs in zip(examples, probabilities):
    spoiler_index = list(model.classes_).index(True)
    print("\nReview:", review)
    print("Spoiler probability:", round(probs[spoiler_index] * 100, 2), "%")



Review: The acting was amazing and the soundtrack was beautiful.
Spoiler probability: 30.36 %

Review: At the end of the movie, the detective discovers that his best friend was the killer.
Spoiler probability: 48.34 %


In [ ]:

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    predictions,
    labels=[False, True]
)

print("Confusion Matrix:")
print(cm)

print("\nActual spoiler reviews:", sum(y_test == True))
print("Correctly detected spoilers:", cm[1][1])
print("Missed spoilers:", cm[1][0])


Confusion Matrix:
[[189  61]
 [ 85 165]]

Actual spoiler reviews: 250
Correctly detected spoilers: 165
Missed spoilers: 85


In [ ]:

import os

print("Model file exists:", os.path.exists("spoiler_model.pkl"))
print("Model variable exists:", "model" in globals())


Model file exists: False
Model variable exists: True


In [ ]:

import joblib

joblib.dump(model, "spoiler_model.pkl")

print("Model saved successfully!")


Model saved successfully!


In [ ]:

from google.colab import files

files.download("spoiler_model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os

print("Model saved:", os.path.exists("spoiler_model.pkl"))


Model saved: True


In [ ]:

%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
import joblib

app = FastAPI(title="Spoiler Alert API")

# Load our trained AI model
model = joblib.load("spoiler_model.pkl")

# Define what users can send
class Review(BaseModel):
    text: str

# Simple health check
@app.get("/")
def home():
    return {"message": "Spoiler Alert API is running!"}

# Predict whether a review contains spoilers
@app.post("/predict")
def predict(review: Review):
    prediction = model.predict([review.text])[0]

    probability = model.predict_proba(
        [review.text]
    )[0]

    spoiler_index = list(model.classes_).index(True)

    return {
        "review": review.text,
        "is_spoiler": bool(prediction),
        "spoiler_probability": round(
            float(probability[spoiler_index]), 4
        )
    }


Writing app.py


In [ ]:

!pip install fastapi uvicorn


In [ ]:

import os

print("API file exists:", os.path.exists("app.py"))
print("Model file exists:", os.path.exists("spoiler_model.pkl"))


API file exists: True
Model file exists: True


In [ ]:

import subprocess
import time

# Start our API server
server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=open("/content/api.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(3)

print("Server running:", server.poll() is None)


Server running: True


In [ ]:

import requests

url = "http://127.0.0.1:8000/predict"

review = {
    "text": "The acting was amazing and the soundtrack was beautiful."
}

response = requests.post(url, json=review, timeout=10)

print("Status code:", response.status_code)
print("AI prediction:", response.json())


Status code: 200
AI prediction: {'review': 'The acting was amazing and the soundtrack was beautiful.', 'is_spoiler': False, 'spoiler_probability': 0.3036}


In [ ]:

# Show the API server error
with open("/content/api.log", "r") as file:
    print(file.read())


INFO:     Started server process [6909]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:59284 - "POST /predict HTTP/1.1" 200 OK



In [ ]:

import requests

try:
    response = requests.post(
        "http://127.0.0.1:8000/predict",
        json={
            "text": "The movie had amazing acting and beautiful music."
        },
        timeout=10
    )

    print("Status:", response.status_code)
    print("Prediction:", response.json())

except requests.exceptions.RequestException as e:
    print("API connection error:", e)


Status: 200
Prediction: {'review': 'The movie had amazing acting and beautiful music.', 'is_spoiler': False, 'spoiler_probability': 0.3925}


In [ ]:

%%writefile requirements.txt
fastapi
uvicorn
scikit-learn
joblib


Writing requirements.txt


In [ ]:

%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY spoiler_model.pkl .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]


Writing Dockerfile


In [ ]:

import os

files_needed = [
    "app.py",
    "spoiler_model.pkl",
    "requirements.txt",
    "Dockerfile"
]

for file in files_needed:
    print(file, ":", os.path.exists(file))


app.py : True
spoiler_model.pkl : True
requirements.txt : True
Dockerfile : True


In [ ]:

from google.colab import files
import zipfile

with zipfile.ZipFile("Spoiler_Alert_Project.zip", "w") as zipf:
    zipf.write("app.py")
    zipf.write("spoiler_model.pkl")
    zipf.write("requirements.txt")
    zipf.write("Dockerfile")

files.download("Spoiler_Alert_Project.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>